In [ ]:
%matplotlib inline

import cv2
from matplotlib import pyplot as plt
import torch
import os
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
import mlflow

from testing_logic_d2 import TestingLogicD2

torch.cuda.is_available()
torch.cuda.device_count()

In [ ]:
host = "127.0.0.1"
port = "8080"
mlflow.set_tracking_uri(uri=F"http://{host}:{port}")

In [ ]:
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(weights=weights, box_score_thresh=0.7)
_ = model.eval()

In [ ]:
model_type = "ResNet"
model_use = "Car Detection"
hardware = "GPU"
model_name = "FasterRCNN_ResNet50_FPN_V2_Weights"
dataset = "combined"
resize_images = False
resize_with_padding = False

In [ ]:
preprocess = weights.transforms()

In [ ]:
def run_experiment(model, params, exp_tags,preprocess, exp_name, debug = False): 
    
    mlflow.set_experiment(exp_name)

    testing_logic_d2 = TestingLogicD2()
    metrics = testing_logic_d2.test_model(model, params=params, preprocess=preprocess, debug=debug)
    
    num_samples = 326
    if dataset == 'train':
        num_samples = testing_logic_d2.num_samples_train
    elif dataset == 'test':
        num_samples = testing_logic_d2.num_samples_test
    experiment_desc = F"Dataset: Dataset_2/{params['dataset']}.\nModel Use: {model_use} \nNumber of Samples: {num_samples} "

    run_name = F"{params['model_name']}-{params['dataset']}-ccm:{params['car_choice_metric']}"
    print(run_name)
    # Start an MLflow run
    with mlflow.start_run(run_name=run_name, description=experiment_desc,tags=exp_tags):
        # Log the hyperparameters
        mlflow.log_params(params)

        # Log the metrics
        for key,val in metrics.items():
            mlflow.log_metric(key, val)


In [ ]:
exp_name = F"{model_use}. Dataset2. {model_type} Only. {hardware}. FasterRCNN_ResNet50"

# for cc_metric in ['confidence','area']:
for cc_metric in ['area']:

    car_choice_metric = cc_metric

    params = {
        "resize_images": resize_images,
        "resize_with_padding": resize_with_padding,
        'model_name':model_name,
        "car_choice_metric": car_choice_metric,
        "model_type": model_type,
        "dataset": dataset
    }
    experiment_tags = {
        "dataset": F"Dataset_2/{dataset}",
        "model_use": model_use,
        "hardware": F"{hardware}"
    }
    print(params)
    print(experiment_tags)
    print("\n")

    
    run_experiment(model=model,
                params=params,
                exp_tags=experiment_tags,
                preprocess=preprocess,
                exp_name=exp_name,
                debug=True
                )
